# Orbit — LoRA fine-tune Gemma 3 270M

Before running:
1. Runtime → Change runtime type → **GPU** (T4 works; L4/A100 is faster and uses bf16).
2. Accept the Gemma license at https://huggingface.co/google/gemma-3-270m-it and add your token as a Colab secret named `HF_TOKEN` (key icon in the left sidebar).
3. Upload to Google Drive, keeping the repo layout:
   - `MyDrive/orbit/lora_fine_tuning_orbit_model/` — `script.py`, `orbit_schema.py`, `generate.py`, `export_litert.py`, `orbit_samples.jsonl`
   - `MyDrive/orbit/dataset/` — `orbit_bubbles_v2.clean.train.jsonl`, `orbit_bubbles_v2.clean.val.jsonl`

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
!rm -rf /content/orbit && cp -r /content/drive/MyDrive/orbit /content/orbit
%cd /content/orbit/lora_fine_tuning_orbit_model
!ls ../dataset

In [ ]:
!pip install -q -U transformers peft accelerate

## Train
Loss is computed on the JSON reply only. The best epoch by validation loss is kept, then the script generates on the validation set and writes `eval.json`.

In [ ]:
!python script.py --epochs 3 --batch 16 --lr 2e-4 --rank 16

In [ ]:
import json
metrics = json.load(open('runs/orbit-lora/eval.json'))
print(json.dumps({k: v for k, v in metrics.items() if k != 'failures'}, indent=2))
for failure in metrics['failures'][:10]:
    print('
---', failure['problem'], '
', failure['entry'], '
>>', failure['reply'])

## Try your own entry

In [ ]:
import torch, json
from transformers import AutoModelForCausalLM, AutoTokenizer
from orbit_schema import user_message
tok = AutoTokenizer.from_pretrained('runs/orbit-lora/merged')
model = AutoModelForCausalLM.from_pretrained('runs/orbit-lora/merged', dtype=torch.float32).to('cuda')
entry = "long day at work and my back hurts. gotta call grandma tmrw its her birthday"
prompt = tok.apply_chat_template([{'role': 'user', 'content': user_message(entry)}], tokenize=False, add_generation_prompt=True)
ids = tok(prompt, return_tensors='pt', add_special_tokens=False).to('cuda')
out = model.generate(**ids, max_new_tokens=512, do_sample=False)
print(tok.decode(out[0, ids['input_ids'].shape[1]:], skip_special_tokens=True))

## Export to `.litertlm` and save to Drive
If installing `litert-torch` changes the torch version, restart the runtime and rerun only the next cell (the merged model is already on disk; copy it to Drive first if you're worried about losing the runtime).

In [ ]:
!cp -r runs/orbit-lora /content/drive/MyDrive/orbit/run-$(date +%Y%m%d-%H%M)-pre-export
!pip install -q litert-torch
!python export_litert.py --merged runs/orbit-lora/merged --out runs/orbit-lora/litert
!cp -r runs/orbit-lora/litert /content/drive/MyDrive/orbit/litert-$(date +%Y%m%d-%H%M)